In [1]:
"""
Complete evaluation of forecast against different datasets
"""
# <eval_forecast.py>
# Parameters
eval_start = "2025-07-02"
eval_end = "2025-07-30"
dataset_choice = "imerg"
eval_vars = "total_precipitation_6hr"
apath = "/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_025_cache/"
params_path_old = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_0.25_37.npz'

params_path_new1 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28_new.npz'
params_path_new2 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28.npz'
norms_dir = '/Datastorage/saptarishi.dhanuka_asp25/norms_gc/'
plots_dir = 'plots/evals'
latmin, latmax, lonmin, lonmax = 6, 38, 35, 65
plot_timesteps = 7
output_pred_old_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
output_pred_finetuned_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
region_vise = True
regions = ['Central_Northeast', 'Hilly_Regions', 'Northeast', 'Northwest', 'South_Peninsular', 'West_Central']
world_regions = ['India']

"""
Complete evaluation of forecast against different datasets with rainfall analysis
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
import logging
import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
import time
import zarr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap

import jax
import optax


# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.62'
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# import utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot, compute_difference_with_targets_sims, plot_sample_from_ds
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
from utils import regrid_hres_fine_to_coarse, generate_sample_era5_dataset, grads_fn, parse_args, process_to_graphcast_format, compute_mse, compute_mse_diffs, mask_dbase_india_buffer, mask_dbase_regions

sys.path.append('/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import trainer.dataloader
from dist_utils import construct_era5_imerg, construct_era5_imerg_6hourly
from datetime import datetime

print("Imports done")


Imports done


In [2]:
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9' 
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

In [3]:
apath = '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_025_cache/'
dbase , _ , _= trainer.dataloader.open_databases(apath,None)
dbase

Found 1 files in /Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_025_cache/
Concatenating monthly databases


<xarray.Dataset> Size: 116GB
Dimensions:                  (time: 123, latitude: 721, longitude: 1440,
                              level: 37)
Coordinates:
  * latitude                 (latitude) float64 6kB -90.0 -89.75 ... 89.75 90.0
  * level                    (level) uint64 296B 1 2 3 5 7 ... 925 950 975 1000
  * longitude                (longitude) float64 12kB 0.0 0.25 ... 359.5 359.8
    number                   int64 8B ...
  * time                     (time) datetime64[ns] 984B 2025-07-01 ... 2025-0...
Data variables: (12/13)
    10m_u_component_of_wind  (time, latitude, longitude) float32 511MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind  (time, latitude, longitude) float32 511MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    2m_temperature           (time, latitude, longitude) float32 511MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    geopotential             (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    geopotential_at_surface  (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    land_sea_mask            (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    ...                       ...
    specific_humidity        (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    temperature              (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    total_precipitation_6hr  (time, latitude, longitude) float32 511MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    u_component_of_wind      (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    v_component_of_wind      (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    vertical_velocity        (time, level, latitude, longitude) float32 19GB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-03T08:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [4]:
# Load a slightly larger window to ensure we have the day before the start_date for initialization
load_start_date = pd.to_datetime(eval_start) - pd.Timedelta(days=1)
load_start_date

Timestamp('2025-07-01 00:00:00')

In [5]:
eval_time_ds = dbase.sel(time=slice(load_start_date.strftime('%Y-%m-%d'), eval_end))
del dbase
select_time_eval = process_to_graphcast_format(eval_time_ds)
select_time_eval

<xarray.Dataset> Size: 113GB
Dimensions:                  (batch: 1, time: 120, lat: 721, lon: 1440,
                              level: 37)
Coordinates:
  * lat                      (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) uint64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    number                   int64 8B 0
    datetime                 (batch, time) datetime64[ns] 960B 2025-07-01 ......
  * time                     (time) timedelta64[ns] 960B 0 days 00:00:00 ... ...
Dimensions without coordinates: batch
Data variables: (12/13)
    10m_u_component_of_wind  (batch, time, lat, lon) float32 498MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind  (batch, time, lat, lon) float32 498MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    2m_temperature           (batch, time, lat, lon) float32 498MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    geopotential             (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    geopotential_at_surface  (batch, lat, lon) float32 4MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    land_sea_mask            (batch, lat, lon) float32 4MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ...                       ...
    specific_humidity        (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    temperature              (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    total_precipitation_6hr  (batch, time, lat, lon) float32 498MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    u_component_of_wind      (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    v_component_of_wind      (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    vertical_velocity        (batch, time, level, lat, lon) float32 18GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-03T08:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [6]:
with open(params_path_old, 'rb') as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
params = ckpt.params

# with open(params_path_new1, "rb") as f:
#     new_params1 = checkpoint.load(f, graphcast.CheckPoint).params
# with open(params_path_new2, "rb") as f:
#     new_params2 = checkpoint.load(f, graphcast.CheckPoint).params

logging.info("Loading models and normalization stats...")
with open('/Datastorage/saptarishi.dhanuka_asp25/norms_gc/diffs_stddev_by_level.nc', 'rb') as f:
    diffs_stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/norms_gc/stddev_by_level.nc', 'rb') as f:
    stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/norms_gc/mean_by_level.nc', 'rb') as f:
    mean_by_level = xr.load_dataset(f).compute()

# --- JAX Function Setup ---
logging.info("Setting up JAX functions...")
state = {}
model_config = ckpt.model_config
task_config = ckpt.task_config
setup_jax_functions.update_configs({
    'params': ckpt.params, 'state': state, 'model_config': ckpt.model_config, 'task_config': task_config,
    'mean_by_level': mean_by_level, 'stddev_by_level': stddev_by_level, 'diffs_stddev_by_level': diffs_stddev_by_level
})
run_forward_jitted = setup_jax_functions.drop_state(setup_jax_functions.with_params(jax.jit(setup_jax_functions.with_configs(
    setup_jax_functions.run_forward.apply))))
jax.config.update("jax_enable_x64", True)

def run_model(params, state, inputs, targets_template, forcings):
    return run_forward_jitted(
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets_template,
        forcings=forcings,
        params=params,
        state=state
    )


In [7]:
initialization_dates = pd.to_datetime(pd.date_range(start=eval_start, end=eval_end, freq='D'))
init_date = initialization_dates[0]
init_date

Timestamp('2025-07-02 00:00:00')

In [8]:
start_slice = init_date - pd.Timedelta(hours=6)
end_slice = init_date + pd.Timedelta(days=7)

start_slice -= select_time_eval.datetime.values[0][0]
end_slice -= select_time_eval.datetime.values[0][0]

# Use .copy(deep=True) to avoid memory issues with repeated slicing
eval_sim_data = select_time_eval.sel(time=slice(start_slice, end_slice)).copy(deep=True)

if eval_sim_data.sizes["time"] < 2:
    raise ValueError("Not enough data in the selected time range for evaluation.")
eval_sim_data

<xarray.Dataset> Size: 28GB
Dimensions:                  (batch: 1, time: 30, lat: 721, lon: 1440, level: 37)
Coordinates:
  * lat                      (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) uint64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    number                   int64 8B 0
    datetime                 (batch, time) datetime64[ns] 240B 2025-07-01T18:...
  * time                     (time) timedelta64[ns] 240B 0 days 18:00:00 ... ...
Dimensions without coordinates: batch
Data variables: (12/13)
    10m_u_component_of_wind  (batch, time, lat, lon) float32 125MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind  (batch, time, lat, lon) float32 125MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    2m_temperature           (batch, time, lat, lon) float32 125MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    geopotential             (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    geopotential_at_surface  (batch, lat, lon) float32 4MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    land_sea_mask            (batch, lat, lon) float32 4MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ...                       ...
    specific_humidity        (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    temperature              (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    total_precipitation_6hr  (batch, time, lat, lon) float32 125MB dask.array<chunksize=(1, 1, 721, 1440), meta=np.ndarray>
    u_component_of_wind      (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    v_component_of_wind      (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
    vertical_velocity        (batch, time, level, lat, lon) float32 5GB dask.array<chunksize=(1, 1, 37, 721, 1440), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-03T08:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [9]:
MAX_FORECAST_STEPS = 28
target_lead_times_str = f"{(MAX_FORECAST_STEPS) * 6}h" # +1 to be safe with slicing
target_lead_times_slice = slice("6h", target_lead_times_str)

In [10]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    eval_sim_data = select_time_eval.sel(time=slice(start_slice, end_slice)).compute()

# 2. Prepare inputs, targets, and forcings for a 7-day rollout
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    eval_sim_data, target_lead_times=target_lead_times_slice, **dataclasses.asdict(task_config))


targets_template = eval_targets * np.nan
ground_truth_var = eval_targets[eval_vars]
eval_inputs

[########################################] | 100% Completed | 12.45 s


<xarray.Dataset> Size: 2GB
Dimensions:                       (batch: 1, time: 2, lat: 721, lon: 1440,
                                   level: 37)
Coordinates:
  * lat                           (lat) float64 6kB -90.0 -89.75 ... 89.75 90.0
  * level                         (level) uint64 296B 1 2 3 5 ... 950 975 1000
  * lon                           (lon) float64 12kB 0.0 0.25 ... 359.5 359.8
    number                        int64 8B 0
  * time                          (time) timedelta64[ns] 16B -1 days +18:00:0...
Dimensions without coordinates: batch
Data variables: (12/18)
    2m_temperature                (batch, time, lat, lon) float32 8MB 216.7 ....
    mean_sea_level_pressure       (batch, time, lat, lon) float32 8MB 1.001e+...
    10m_v_component_of_wind       (batch, time, lat, lon) float32 8MB 1.772 ....
    10m_u_component_of_wind       (batch, time, lat, lon) float32 8MB 6.055 ....
    total_precipitation_6hr       (batch, time, lat, lon) float32 8MB 9.537e-...
    temperature                   (batch, time, level, lat, lon) float32 307MB ...
    ...                            ...
    year_progress_sin             (batch, time) float32 8B 0.003295 -0.001006
    year_progress_cos             (batch, time) float32 8B -1.0 -1.0
    day_progress_sin              (batch, time, lon) float32 12kB -1.0 ... -0...
    day_progress_cos              (batch, time, lon) float32 12kB 1.192e-08 ....
    geopotential_at_surface       (batch, lat, lon) float32 4MB 2.772e+04 ......
    land_sea_mask                 (batch, lat, lon) float32 4MB 1.0 1.0 ... 0.0
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-03T08:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [11]:
test_preds = run_model(params, state, eval_inputs, targets_template, eval_forcings)
test_preds

<xarray.Dataset> Size: 26GB
Dimensions:                  (time: 28, batch: 1, lat: 721, lon: 1440, level: 37)
Coordinates:
  * time                     (time) timedelta64[ns] 224B 0 days 06:00:00 ... ...
  * lat                      (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) uint64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    number                   int64 8B 0
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 116MB xarray_jax...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 116MB xarray_jax...
    2m_temperature           (time, batch, lat, lon) float32 116MB xarray_jax...
    geopotential             (time, batch, level, lat, lon) float32 4GB xarra...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 116MB xarray_jax...
    specific_humidity        (time, batch, level, lat, lon) float32 4GB xarra...
    temperature              (time, batch, level, lat, lon) float32 4GB xarra...
    total_precipitation_6hr  (time, batch, lat, lon) float32 116MB xarray_jax...
    u_component_of_wind      (time, batch, level, lat, lon) float32 4GB xarra...
    v_component_of_wind      (time, batch, level, lat, lon) float32 4GB xarra...
    vertical_velocity        (time, batch, level, lat, lon) float32 4GB xarra...

In [20]:
eval_targets

<xarray.Dataset> Size: 26GB
Dimensions:                  (batch: 1, time: 28, lat: 721, lon: 1440, level: 37)
Coordinates:
  * lat                      (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * level                    (level) uint64 296B 1 2 3 5 7 ... 925 950 975 1000
  * lon                      (lon) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    number                   int64 8B 0
  * time                     (time) timedelta64[ns] 224B 0 days 06:00:00 ... ...
Dimensions without coordinates: batch
Data variables:
    2m_temperature           (batch, time, lat, lon) float32 116MB 216.7 ... ...
    mean_sea_level_pressure  (batch, time, lat, lon) float32 116MB 9.989e+04 ...
    10m_v_component_of_wind  (batch, time, lat, lon) float32 116MB -0.5834 .....
    10m_u_component_of_wind  (batch, time, lat, lon) float32 116MB 5.613 ... ...
    total_precipitation_6hr  (batch, time, lat, lon) float32 116MB 4.768e-07 ...
    temperature              (batch, time, level, lat, lon) float32 4GB 230.5...
    geopotential             (batch, time, level, lat, lon) float32 4GB 4.03e...
    u_component_of_wind      (batch, time, level, lat, lon) float32 4GB -3.79...
    v_component_of_wind      (batch, time, level, lat, lon) float32 4GB -13.3...
    vertical_velocity        (batch, time, level, lat, lon) float32 4GB 9.293...
    specific_humidity        (batch, time, level, lat, lon) float32 4GB 3.78e...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-03T08:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [21]:
with ProgressBar():
    eval_targets['total_precipitation_6hr'].to_netcdf('/Datastorage/saptarishi.dhanuka_asp25/test_025_2025aug_targs.nc', mode='w')

In [22]:
with ProgressBar():
    test_preds['total_precipitation_6hr'].to_netcdf('/Datastorage/saptarishi.dhanuka_asp25/test_025_2025aug_preds.nc', mode='w')